In [1]:
import pandas as pd
import numpy as np
import re
import string
import warnings
import emoji
from langdetect import detect, DetectorFactory
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
DetectorFactory.seed = 0 # for consistency

## Data Cleaning

Dataset needs to be cleaned to engineer features, extract clusters, and prepare data for modeling.

In [2]:
# Read and load dataset
df = pd.read_csv('data/amazon_bestsellers_reviews.csv')
print(df.dtypes, '\n') # helpful_votes should be int, date should be datetime64
print(df.isna().sum()) # impute null helpful_votes
df.head()

department       object
product_index     int64
product_name     object
product_url      object
reviewer         object
rating            int64
date             object
verified         object
title            object
body             object
helpful_votes    object
variant          object
image_count       int64
dtype: object 

department         0
product_index      0
product_name       0
product_url        0
reviewer           0
rating             0
date               0
verified           0
title              0
body               0
helpful_votes    860
variant           97
image_count        0
dtype: int64


,department,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count
0,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,"Reviewed in the United States on March 17, 2026",Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28 people found this helpful,Size: One SizeStyle: USB-C,0
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,"Reviewed in the United States on April 11, 2026",Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8 people found this helpful,Size: One SizeStyle: USB-C,0
2,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,"Reviewed in the United States on April 10, 2026",Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Let’s be honest: nobody actually wants to live...,7 people found this helpful,Size: One SizeStyle: USB-C,0
3,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,"Reviewed in the United States on March 17, 2026",Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23 people found this helpful,Size: One SizeStyle: USB-C,0
4,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,"Reviewed in the United States on April 25, 2026",Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",NaN,Size: One SizeStyle: USB-C,0


In [3]:
# Text Preprocessing (Title + Body)

print('Rows before filtering:', df.shape[0])

# Filter non-english reviews
def is_english(text):
    try:
        return detect(str(text)) == 'en'
    except:
        return False

df = df[df['body'].apply(is_english)].copy()
df = df.reset_index(drop=True)

# Decode emojis and remove non-ascii characters
df['title'] = df['title'].apply(lambda x: emoji.demojize(str(x)))
df['body'] = df['body'].apply(lambda x: emoji.demojize(str(x)))
df['title'] = df['title'].str.encode('ascii', 'ignore').str.decode('ascii')
df['body'] = df['body'].str.encode('ascii', 'ignore').str.decode('ascii')

# Preparing Review Text for Processing (TF-IDF, LSA)
df['text_for_lsa'] = df['title'].fillna('') + ' ' + df['body'].fillna('')
df['text_for_lsa'] = df['text_for_lsa'].str.lower()
df['text_for_lsa'] = df['text_for_lsa'].str.replace(f"[{re.escape(string.punctuation)}]", " ", regex=True)
df['text_for_lsa'] = df['text_for_lsa'].str.replace(r'\d+', '', regex=True)

# Removing Stopwords
def remove_stopwords(text):
    words = [word for word in text.split() if word not in ENGLISH_STOP_WORDS]
    return ' '.join(words)

df['text_for_lsa'] = df['text_for_lsa'].apply(remove_stopwords)
df['text_for_lsa'] = df['text_for_lsa'].str.strip().str.replace(r'\s+', ' ', regex=True)

print('Rows after filtering:', df.shape[0])
display(df.head())

Rows before filtering: 1784
Rows after filtering: 1513


,department,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count,text_for_lsa
0,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,"Reviewed in the United States on March 17, 2026",Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28 people found this helpful,Size: One SizeStyle: USB-C,0,best class quality sound apple earpods usb c t...
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,"Reviewed in the United States on April 11, 2026",Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8 people found this helpful,Size: One SizeStyle: USB-C,0,great sound comfort bought son prefers traditi...
2,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,"Reviewed in the United States on April 10, 2026",Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Lets be honest: nobody actually wants to live ...,7 people found this helpful,Size: One SizeStyle: USB-C,0,audiophile secret small dongle big practicalit...
3,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,"Reviewed in the United States on March 17, 2026",Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23 people found this helpful,Size: One SizeStyle: USB-C,0,better overhead headphones honestly love headp...
4,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,"Reviewed in the United States on April 25, 2026",Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",NaN,Size: One SizeStyle: USB-C,0,worth extra money best ones best buy usbc ear ...


## Feature Engineering & Target Creation

Before progressing to dimensionality reduction and modeling, we need to transform our raw text and categorical data into a rich set of numerical features. This step establishes our target variable, prevents data leakage, and extracts quantitative signals from the reviews.

* **Target Creation & Leakage Prevention:** We parse the raw `helpful_votes` strings to create our binary target column `is_helpful` (1 if votes > 0, else 0). *Crucially*, the original vote count columns are kept strictly separate from the final feature set to prevent target leakage, ensuring the model cannot "cheat" by looking at the actual vote counts during training.
* **Categorical Encoding:** The `department` column is One-Hot Encoded (`pd.get_dummies`), converting it into a machine-readable format so the model can learn category-specific review behaviors.
* **Rating Extremity:** We calculate `rating_mid_dist` (the absolute distance from a median 3-star rating) to capture how "extreme" a review is, as highly polarized reviews often attract more helpful votes than neutral ones.
* **Effort Metrics:** We extract structural text features (`title_len`, `body_len`, `word_count`, `avg_word_len`). These serve as mathematical proxies for the reviewer's effort, detail, and complexity.
* **Sentiment Analysis (VADER):** We utilize **VADER (Valence Aware Dictionary and sEntiment Reasoner)** to quantify the emotional tone of the review body. VADER is specifically optimized for informal, internet-based text, making it highly effective at interpreting capitalization, punctuation (e.g., "GREAT!!!"), and slang common in Amazon reviews.
  * *Why we use it:* Instead of just taking the raw sentiment score, we split it into two engineered features: `sentiment_is_positive` (the binary direction of the emotion) and `sentiment_strength` (the absolute magnitude of the emotion). This is a deliberate choice because an intensely angry 1-star review and a glowing 5-star review both demonstrate high emotional investment/strength, which is a strong independent predictor of a review being voted as "helpful."

In [4]:
# Parse helpful_votes and cast into int; create target column is_helpful
pattern = r'(One|\d+)'
df['helpful_votes'] = df['helpful_votes'].str.extract(pattern)[0]
df['helpful_votes'] = df['helpful_votes'].replace('One', 1)
df['helpful_votes'] = pd.to_numeric(df['helpful_votes'], errors='coerce').fillna(0).astype(int)
df['is_helpful'] = (df['helpful_votes'] > 0).astype(int) # Assign 1 or 0

# One-hot encode department
df = pd.get_dummies(df, columns=['department'], dtype=int)

# Split date into date and location, parse date
pattern = r'Reviewed in (.+?) on (.+)'
df[['location', 'date']] = df['date'].str.extract(pattern)
df['location'] = df['location'].str.replace(r'(?i)^the\s+', '', regex=True)
df['date'] = pd.to_datetime(df['date'])
df.head(1)

# Rating: already numeric (1–5); cast to int for safety
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(df['rating'].median()).astype(int)

# Distance from median rating (3)
df['rating_mid_dist'] = df['rating'] - 3
df['rating_mid_dist'] = df['rating_mid_dist'].abs()

# Title / Body length features (text remains; lengths are numeric)
df['title_len'] = df['title'].fillna('').str.len().astype(int)
df['body_len'] = df['body'].fillna('').str.len().astype(int)
df['word_count'] = df['text_for_lsa'].str.split().str.len().fillna(0).astype(int)
raw_word_count = df['body'].fillna('').str.split().str.len()
df['avg_word_len'] = df['body_len'] / raw_word_count.replace(0, np.nan) 
df['avg_word_len'] = df['avg_word_len'].fillna(0) # Put 0s back

# Sentiment Analysis
analyzer = SentimentIntensityAnalyzer()
df['sentiment_strength'] = df['body'].fillna('').apply(
    lambda x: analyzer.polarity_scores(x)['compound'])
df['sentiment_is_positive'] = df['sentiment_strength'].apply(
    lambda x: 1 if x > 0 else 0)
df['sentiment_strength'] = df['sentiment_strength'].abs()

print("DataFrame dtypes after encoding:")
print(df.dtypes)
print(f"\nDataFrame shape: {df.shape}")
df

DataFrame dtypes after encoding:
product_index                                    int64
product_name                                    object
product_url                                     object
reviewer                                        object
rating                                           int64
date                                    datetime64[ns]
verified                                        object
title                                           object
body                                            object
helpful_votes                                    int64
variant                                         object
image_count                                      int64
text_for_lsa                                    object
is_helpful                                       int64
department_Clothing, Shoes & Jewelry             int64
department_Electronics                           int64
department_Kitchen & Home                        int64
location                        

,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,...,department_Electronics,department_Kitchen & Home,location,rating_mid_dist,title_len,body_len,word_count,avg_word_len,sentiment_strength,sentiment_is_positive
0,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,2026-03-17,Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28,...,1,0,United States,2,31,557,61,6.258427,0.9876,1
1,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,2026-04-11,Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8,...,1,0,United States,2,23,495,43,6.036585,0.9547,1
2,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,2026-04-10,Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Lets be honest: nobody actually wants to live ...,7,...,1,0,United States,2,56,1369,135,6.251142,0.9857,1
3,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,2026-03-17,Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23,...,1,0,United States,2,31,935,82,5.843750,0.9953,1
4,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,2026-04-25,Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",0,...,1,0,United States,2,51,132,20,5.076923,0.9705,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1508,50,IVAPUPU 2 Pack 6FT Table Cloth for Rectangular...,https://www.amazon.com/dp/B0B56J7ZH8,Alyise Roberge,5,2026-01-14,Verified Purchase,Super snug but look awesome,Great for my folding tables!,0,...,0,1,Australia,2,27,28,7,5.600000,0.6588,1
1509,50,IVAPUPU 2 Pack 6FT Table Cloth for Rectangular...,https://www.amazon.com/dp/B0B56J7ZH8,K. Edwards,5,2025-08-06,Verified Purchase,Good quality,Good quality tablecloths that look classier fo...,0,...,0,1,Australia,2,12,55,8,6.875000,0.4404,1
1510,50,IVAPUPU 2 Pack 6FT Table Cloth for Rectangular...,https://www.amazon.com/dp/B0B56J7ZH8,KellzBellz,5,2024-09-28,Verified Purchase,Great product,Arrived quickly and in perfect condition.,0,...,0,1,Australia,2,13,41,6,6.833333,0.5719,1
1511,50,IVAPUPU 2 Pack 6FT Table Cloth for Rectangular...,https://www.amazon.com/dp/B0B56J7ZH8,Maggie Sullivan,5,2024-09-27,Verified Purchase,Very sturdy,Easy to use,0,...,0,1,Australia,2,11,11,3,3.666667,0.4404,1


## Export Dataset for Mining and Modeling

In [5]:
# Sanity Checks

# Catch categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Remaining categorical columns: {cat_cols if cat_cols else 'None — all numerical ✓'}")

# Catch missing values
print("\nMissing values per column (top offenders):")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  None ✓")

# Catch zero variance columns
zero_var = [col for col in df.columns if df[col].nunique() <= 1]
print("\nZero variance columns:", zero_var if zero_var else "None ✓")

# Class Balance Check
print("\nTarget Distribution:")
print(df['is_helpful'].value_counts(normalize=True).round(3), '\n')

# Fill any edge-case NaNs and infs
df.fillna(0, inplace=True)
df.replace([np.inf, -np.inf], 0, inplace=True)

# Check for remaining NaN or inf values
has_nan = df.isna().any().any()
has_inf = np.isinf(df.select_dtypes(include=[np.number])).any().any()

if has_nan or has_inf:
    print("Warning: Data still contains NaN or inf values!")
    print(f"NaN present: {has_nan}, Inf present: {has_inf}")
else:
    print("No NaN or inf values detected.")

print("\nFinal shape after NaN and inf fill:", df.shape, '\n')

Remaining categorical columns: ['product_name', 'product_url', 'reviewer', 'verified', 'title', 'body', 'variant', 'text_for_lsa', 'location']

Missing values per column (top offenders):
variant    70
dtype: int64

Zero variance columns: ['verified']

Target Distribution:
is_helpful
1    0.611
0    0.389
Name: proportion, dtype: float64 

No NaN or inf values detected.

Final shape after NaN and inf fill: (1513, 25) 



In [6]:
# Preview
print(f"Final dataset shape: {df.shape}")
print(f"\nColumn list ({len(df.columns)} columns):")
print(list(df.columns))
print("\nSample:")
df.head()

Final dataset shape: (1513, 25)

Column list (25 columns):
['product_index', 'product_name', 'product_url', 'reviewer', 'rating', 'date', 'verified', 'title', 'body', 'helpful_votes', 'variant', 'image_count', 'text_for_lsa', 'is_helpful', 'department_Clothing, Shoes & Jewelry', 'department_Electronics', 'department_Kitchen & Home', 'location', 'rating_mid_dist', 'title_len', 'body_len', 'word_count', 'avg_word_len', 'sentiment_strength', 'sentiment_is_positive']

Sample:


,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,...,department_Electronics,department_Kitchen & Home,location,rating_mid_dist,title_len,body_len,word_count,avg_word_len,sentiment_strength,sentiment_is_positive
0,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,2026-03-17,Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28,...,1,0,United States,2,31,557,61,6.258427,0.9876,1
1,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,2026-04-11,Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8,...,1,0,United States,2,23,495,43,6.036585,0.9547,1
2,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,2026-04-10,Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Lets be honest: nobody actually wants to live ...,7,...,1,0,United States,2,56,1369,135,6.251142,0.9857,1
3,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,2026-03-17,Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23,...,1,0,United States,2,31,935,82,5.843750,0.9953,1
4,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,2026-04-25,Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",0,...,1,0,United States,2,51,132,20,5.076923,0.9705,1


In [7]:
# Export final dataset
df.to_csv("data/amazon_bestsellers_reviews_cleaned.csv", index=False)
print("Exported ✓")

Exported ✓
